**Create gold.dim_date**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY full_date) AS date_id,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month,
    DAY(full_date) AS day,
    DAYOFWEEK(full_date) AS day_of_week_num
FROM (
    SELECT DISTINCT CAST(to_timestamp(`movement.scheduledTime.utc`, 'yyyy-MM-dd HH:mmXXX') AS DATE) AS full_date
    FROM silver.flights_clean
) t;

**Create gold.dim_airline**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_airline AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY airline_iata) AS airline_id,
    airline_name,
    airline_iata,
    airline_icao
FROM (
    SELECT 
        `airline.name` AS airline_name, 
        `airline.iata` AS airline_iata, 
        `airline.icao` AS airline_icao,
        ROW_NUMBER() OVER (PARTITION BY `airline.iata` ORDER BY `airline.name`) AS rn
    FROM silver.flights_clean
    WHERE `airline.iata` IS NOT NULL
) t
WHERE rn = 1;

**Create gold.dim_airport**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_airport AS
WITH AllAirports AS (
    SELECT 
        origin_airport_name AS airport_name, 
        origin_airport_iata AS airport_iata, 
        origin_airport_icao AS airport_icao 
    FROM silver.flights_clean 
    WHERE origin_airport_iata IS NOT NULL
    UNION
    SELECT 
        destination_airport_name, 
        destination_airport_iata, 
        destination_airport_icao 
    FROM silver.flights_clean 
    WHERE destination_airport_iata IS NOT NULL
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY airport_iata) AS airport_id,
    airport_name,
    airport_iata,
    airport_icao
FROM AllAirports;

**Create gold.dim_aircraft**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_aircraft AS
-- defualt value as unknown
SELECT 
    -1 AS aircraft_id, 
    'Unknown' AS aircraft_model, 
    'Unknown' AS aircraft_reg, 
    'Unknown' AS aircraft_modeS

UNION ALL

--  Real aircraft (only one row per aircraft_reg)
SELECT 
    ROW_NUMBER() OVER (ORDER BY aircraft_reg) AS aircraft_id,
    aircraft_model, 
    aircraft_reg, 
    aircraft_modeS
FROM (
    SELECT 
        `aircraft.model` AS aircraft_model, 
        `aircraft.reg` AS aircraft_reg, 
        `aircraft.modeS` AS aircraft_modeS,
        ROW_NUMBER() OVER (PARTITION BY `aircraft.reg` ORDER BY `aircraft.model`) AS rn
    FROM silver.flights_clean
    WHERE `aircraft.reg` IS NOT NULL
) t
WHERE rn = 1;

**Create gold.dim_terminal**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_terminal AS
SELECT -1 AS terminal_id, 'Unknown' AS terminal
UNION ALL
SELECT 
    ROW_NUMBER() OVER (ORDER BY terminal) AS terminal_id,
    terminal
FROM (
    SELECT DISTINCT `movement.terminal` AS terminal
    FROM silver.flights_clean
    WHERE `movement.terminal` IS NOT NULL
) t;

**Create gold.dim_status**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_status AS
SELECT -1 AS status_id, 'Unknown' AS status
UNION ALL
SELECT 
    ROW_NUMBER() OVER (ORDER BY status) AS status_id,
    status
FROM (
    SELECT DISTINCT status
    FROM silver.flights_clean
    WHERE status IS NOT NULL 
      AND status != 'Unknown'  -- remove Unknown status
) t;

**Create gold.fact_flights**

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fact_flights AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY f.`movement.scheduledTime.utc`) AS flight_id,
    COALESCE(d.date_id, -1) AS date_id,
    COALESCE(al.airline_id, -1) AS airline_id,
    COALESCE(ac.aircraft_id, -1) AS aircraft_id,
    COALESCE(org.airport_id, -1) AS origin_airport_id,
    COALESCE(dest.airport_id, -1) AS destination_airport_id,
    COALESCE(t.terminal_id, -1) AS terminal_id,
    COALESCE(s.status_id, -1) AS status_id,
    f.flight_number, 
    f.flight_type, 
    f.codeshareStatus, 
    f.isCargo, 
    f.callSign,
    f.`movement.scheduledTime.utc` AS scheduled_time_utc, 
    f.`movement.scheduledTime.local` AS scheduled_time_local
FROM silver.flights_clean f
LEFT JOIN gold.dim_date d 
    ON CAST(f.`movement.scheduledTime.utc` AS DATE) = d.full_date
LEFT JOIN gold.dim_airline al 
    ON f.`airline.iata` = al.airline_iata
LEFT JOIN gold.dim_aircraft ac 
    ON f.`aircraft.reg` = ac.aircraft_reg
LEFT JOIN gold.dim_airport org 
    ON f.origin_airport_iata = org.airport_iata
LEFT JOIN gold.dim_airport dest 
    ON f.destination_airport_iata = dest.airport_iata
LEFT JOIN gold.dim_terminal t 
    ON f.`movement.terminal` = t.terminal
LEFT JOIN gold.dim_status s 
    ON f.status = s.status;

In [0]:
%sql
SELECT 
    'Total Flights' AS metric, COUNT(*) AS count FROM gold.fact_flights
UNION ALL
SELECT 'Flights with Unknown Aircraft', COUNT(*) FROM gold.fact_flights WHERE aircraft_id = -1
UNION ALL
SELECT 'Flights with Unknown Terminal', COUNT(*) FROM gold.fact_flights WHERE terminal_id = -1
UNION ALL
SELECT 'Flights with Unknown Status', COUNT(*) FROM gold.fact_flights WHERE status_id = -1
UNION ALL
SELECT 'Flights with Known Airline', COUNT(*) FROM gold.fact_flights WHERE airline_id != -1;

**Check all dimension tables for duplicates**

In [0]:
%sql
-- 
SELECT 'dim_aircraft' AS dim_table, aircraft_reg AS dup_key, COUNT(*) AS cnt
FROM gold.dim_aircraft
WHERE aircraft_reg != 'Unknown'
GROUP BY aircraft_reg
HAVING COUNT(*) > 1

UNION ALL

SELECT 'dim_airline', airline_iata, COUNT(*)
FROM gold.dim_airline
GROUP BY airline_iata
HAVING COUNT(*) > 1

UNION ALL

SELECT 'dim_airport', airport_iata, COUNT(*)
FROM gold.dim_airport
GROUP BY airport_iata
HAVING COUNT(*) > 1

UNION ALL

SELECT 'dim_terminal', terminal, COUNT(*)
FROM gold.dim_terminal
GROUP BY terminal
HAVING COUNT(*) > 1

UNION ALL

SELECT 'dim_status', status, COUNT(*)
FROM gold.dim_status
GROUP BY status
HAVING COUNT(*) > 1

UNION ALL

SELECT 'dim_date', CAST(full_date AS STRING), COUNT(*)
FROM gold.dim_date
GROUP BY CAST(full_date AS STRING)
HAVING COUNT(*) > 1;

**Check the duplication**

In [0]:
%sql
-- is there is any duplication in airport_iata؟
SELECT airport_iata, COUNT(*) 
FROM gold.dim_airport 
GROUP BY airport_iata 
HAVING COUNT(*) > 1;

-- is there is any duplication in airline_iata؟
SELECT airline_iata, COUNT(*) 
FROM gold.dim_airline 
GROUP BY airline_iata 
HAVING COUNT(*) > 1;

**Check the if there is null value in gold.dim_airport, and gold.dim_airline**

In [0]:
%sql
-- Are there any airports with no name or IATA code?
SELECT * FROM gold.dim_airport 
WHERE airport_id IS NULL 
   OR airport_iata IS NULL 
   OR airport_name IS NULL;

-- Are there any airlines without a code?
SELECT * FROM gold.dim_airline 
WHERE airline_id IS NULL 
   OR airline_iata IS NULL;

**Count the Airlines, Airports and Days**

In [0]:
%sql
SELECT 'Airlines' AS metric, COUNT(*) AS total FROM gold.dim_airline
UNION ALL
SELECT 'Airports', COUNT(*) FROM gold.dim_airport
UNION ALL
SELECT 'Days', COUNT(*) FROM gold.dim_date;

**Top 10 destinations by number of flights**


In [0]:
%sql
-- 
SELECT 
    dest.airport_name AS destination,
    dest.airport_iata,
    COUNT(*) AS total_flights
FROM gold.fact_flights f
JOIN gold.dim_airport dest ON f.destination_airport_id = dest.airport_id
WHERE dest.airport_id != -1  -- Exclude "Unknown
GROUP BY dest.airport_name, dest.airport_iata
ORDER BY total_flights DESC
LIMIT 10;

**Check the value gold.dim_status**

In [0]:
%sql
SELECT 
    s.status,
    COUNT(*) AS total_flights,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM gold.fact_flights f
JOIN gold.dim_status s ON f.status_id = s.status_id
GROUP BY s.status
ORDER BY total_flights DESC;

**Validate the quality of the star schema**

In [0]:
%sql
-- 1. The number of rows in fact_flights should equal the number of rows in silver
SELECT 
    (SELECT COUNT(*) FROM silver.flights_clean) AS silver_count,
    (SELECT COUNT(*) FROM gold.fact_flights) AS gold_count;
-- 

-- 2. Verify that there are no NULLs in the foreign keys
SELECT 
    SUM(CASE WHEN airline_id IS NULL THEN 1 ELSE 0 END) AS null_airline,
    SUM(CASE WHEN origin_airport_id IS NULL THEN 1 ELSE 0 END) AS null_origin,
    SUM(CASE WHEN destination_airport_id IS NULL THEN 1 ELSE 0 END) AS null_dest,
    SUM(CASE WHEN date_id IS NULL THEN 1 ELSE 0 END) AS null_date
FROM gold.fact_flights;


In [0]:
%sql
SELECT 
    dest.airport_name AS destination,
    dest.airport_iata,
    COUNT(*) AS total_flights
FROM gold.fact_flights f
JOIN gold.dim_airport dest ON f.destination_airport_id = dest.airport_id
WHERE dest.airport_id != -1
GROUP BY dest.airport_name, dest.airport_iata
ORDER BY total_flights DESC
LIMIT 10;

**Share of each airline in total flights**

In [0]:
%sql
SELECT 
    al.airline_name,
    COUNT(*) AS total_flights,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS market_share_pct
FROM gold.fact_flights f
JOIN gold.dim_airline al ON f.airline_id = al.airline_id
GROUP BY al.airline_name
ORDER BY total_flights DESC;

**Flight distribution by hour of the day**

In [0]:
%sql
SELECT 
    CAST(SUBSTRING(scheduled_time_local, 12, 2) AS INT) AS hour_of_day,
    COUNT(*) AS total_flights
FROM gold.fact_flights
GROUP BY CAST(SUBSTRING(scheduled_time_local, 12, 2) AS INT)
ORDER BY hour_of_day;

**Flights by day of the week**

In [0]:
%sql
SELECT 
    d.day_of_week_num,
    CASE d.day_of_week_num
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_name,
    COUNT(*) AS total_flights
FROM gold.fact_flights f
JOIN gold.dim_date d ON f.date_id = d.date_id
GROUP BY d.day_of_week_num
ORDER BY d.day_of_week_num;

**Percentage of flights with unknown and Known aircraft**

In [0]:
%sql
SELECT 
    CASE 
        WHEN aircraft_id = -1 THEN 'Unknown'
        ELSE 'Known'
    END AS aircraft_status,
    COUNT(*) AS total_flights,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM gold.fact_flights
GROUP BY aircraft_id = -1;